---
title: "深度优先搜索 (DFS)——网格搜索"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [2]:
from collections import deque
from typing import List
import copy

## 📝 题目 200：岛屿数量

::: {.callout-caution}
### 题目要求
**描述**：
给你一个由 `'1'`（陆地）和 `'0'`（水）组成的二维网格 `grid`，请你计算网格中岛屿的数量。

**岛屿定义**：
岛屿总是被水包围，并且每座岛屿只能由水平方向和/或竖直方向上相邻的陆地连接形成。此外，你可以假设该网格的四条边都被水包围。

**输入输出示例**：

* **示例 1**：
    * **输入**：
      ```text
      grid = [
        ["1","1","1","1","0"],
        ["1","1","0","1","0"],
        ["1","1","0","0","0"],
        ["0","0","0","0","0"]
      ]
      ```
    * **输出**：`1`

* **示例 2**：
    * **输入**：
      ```text
      grid = [
        ["1","1","0","0","0"],
        ["1","1","0","0","0"],
        ["0","0","1","0","0"],
        ["0","0","0","1","1"]
      ]
      ```
    * **输出**：`3`
:::

In [9]:
class Solution200:
    def numIslands(self, grid: List[List[str]]) -> int:
        if not grid:
            return 0
        rows, cols = len(grid), len(grid[0])
        count = 0

        def dfs(r: int, c: int) -> None:
            if 0 <= r < rows and 0 <= c < cols and grid[r][c] == '1':  # 只有是陆地且不越界，才干活
                grid[r][c] = '0'  # 淹没当前陆地 (防止重复访问)
                dfs(r - 1, c)
                dfs(r + 1, c)
                dfs(r, c - 1)
                dfs(r, c + 1)

        for i in range(rows):
            for j in range(cols):
                if grid[i][j] == '1':
                    count += 1  # 发现新岛屿
                    dfs(i, j)  # 派出拆迁队，把这个岛屿所有的陆地都标记掉
        return count

In [10]:
#| code-fold: true
def test_200(func):
    test_cases = [
        {"desc": "标准单一大岛", "grid": [["1","1","1"],["1","1","1"],["1","1","1"]], "expected": 1},
        {"desc": "四个对角小岛", "grid": [["1","0","1"],["0","0","0"],["1","0","1"]], "expected": 4},
        {"desc": "全是水", "grid": [["0","0"],["0","0"]], "expected": 0},
        {"desc": "全是陆地", "grid": [["1","1"],["1","1"]], "expected": 1},
        {"desc": "L型岛屿", "grid": [["1","0","0"],["1","0","0"],["1","1","1"]], "expected": 1},
        {"desc": "只有一格陆地", "grid": [["1"]], "expected": 1},
        {"desc": "空网格", "grid": [], "expected": 0},
        {"desc": "棋盘状岛屿", "grid": [["1","0","1","0"],["0","1","0","1"],["1","0","1","0"]], "expected": 6},
        {"desc": "长条型岛屿（水平）", "grid": [["1","1","1","1","1"]], "expected": 1},
        {"desc": "长条型岛屿（垂直）", "grid": [["1"],["1"],["1"],["1"]], "expected": 1}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        grid_copy = copy.deepcopy(tc["grid"])
        actual = func(grid_copy)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")
    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_200(Solution200().numIslands)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 1    | 1    | 标准单一大岛
2    | ✅ PASS | 4    | 4    | 四个对角小岛
3    | ✅ PASS | 0    | 0    | 全是水
4    | ✅ PASS | 1    | 1    | 全是陆地
5    | ✅ PASS | 1    | 1    | L型岛屿
6    | ✅ PASS | 1    | 1    | 只有一格陆地
7    | ✅ PASS | 0    | 0    | 空网格
8    | ✅ PASS | 6    | 6    | 棋盘状岛屿
9    | ✅ PASS | 1    | 1    | 长条型岛屿（水平）
10   | ✅ PASS | 1    | 1    | 长条型岛屿（垂直）
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

在网格里做 DFS，最怕的是“死循环”（比如 A 看到 B 是陆地，B 又回头看到 A 是陆地）。为了解决这个问题，我们要么用 `visited` 集合，要么用更酷的 **“原地淹没”法**。

**1. 巡逻兵（主循环）**：
   - 我们像巡逻兵一样，从左到右、从上到下遍历整个地图。
   - 只要看到 `'1'`，说明发现了一个新岛屿！岛屿计数器 `+1`。
   - **关键动作**：为了不重复计算这个岛屿，我们立刻派出“拆迁队”（DFS）把这一整块连通的陆地全部变成 `'0'`（变成水）。

**2. 拆迁队（DFS 递归函数）**：
   - **任务**：给我一个坐标 `(r, c)`，如果是陆地，就把它淹了，并去它的上下左右继续淹。
   - **递归出口 (Base Case)**：
     - 坐标出界了（走出地图）。
     - 这里的地已经是水了 (`'0'`)。
   - **具体动作**：
     - 把当前格子改成 `'0'`。
     - 向 **上、下、左、右** 四个方向发动递归。

**3. 为什么不用担心死循环？**
   - 因为我们每走过一个地方就把 `'1'` 变成了 `'0'`，陆地越来越少，递归最终一定会停下来。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N)$。其中 $M$ 是行数，$N$ 是列数。每个格子最多被访问一次。
* **空间复杂度**：$O(M \times N)$。最坏情况下（全是陆地），递归栈的深度会达到地图的总面积。
:::

## 📝 题目 695：岛屿的最大面积 (Max Area of Island)

::: {.callout-caution}
### 题目要求
**描述**：
给你一个大小为 `m x n` 的二进制矩阵 `grid` 。
- `'1'` 表示陆地，`'0'` 表示水。
- **岛屿** 是由相邻的 `'1'`（至少水平或垂直方向相邻）连接形成的组合。
- 假设矩阵外全被水包围。

**返回目标**：
- 计算并返回 `grid` 中任意岛屿的 **最大面积**。
- 如果没有岛屿，则返回面积为 `0`。

**输入输出示例**：
* **示例 1**：
    * **输入**：
      ```text
      [[0,0,1,0,0,0,0,1,0,0,0,0,0],
       [0,0,0,0,0,0,0,1,1,1,0,0,0],
       [0,1,1,0,1,0,0,0,0,0,0,0,0],
       [0,1,0,0,1,1,0,0,1,0,1,0,0],
       [0,1,0,0,1,1,0,0,1,1,1,0,0]]
      ```
    * **输出**：`6`
    * **解释**：图中有一个岛屿的面积是 6（由 6 个 '1' 组成）。

* **示例 2**：
    * **输入**：`grid = [[0,0,0,0,0,0,0,0]]`
    * **输出**：`0`
:::

In [31]:
class Solution695:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        if not grid:
            return 0
        rows, cols = len(grid), len(grid[0])
        max_area = 0

        def dfs(r: int, c: int) -> int:
            if 0 <= r < rows and 0 <= c < cols and grid[r][c] == 1:  # 只有是陆地时才进入逻辑
                grid[r][c] = 0  # 淹没它，防止复读
                current_area = 1
                current_area += dfs(r - 1, c)
                current_area += dfs(r + 1, c)
                current_area += dfs(r, c + 1)
                current_area += dfs(r, c - 1)
                return current_area
            return 0  # 如果不是陆地或是越界，贡献的面积就是 0

        for i in range(rows):
            for j in range(cols):
                if grid[i][j] == 1:
                    max_area = max(max_area, dfs(i, j))  # 每发现一处陆地，就更新全局最大面积
        return max_area

In [32]:
#| code-fold: true
def test_695(func):
    test_cases = [
        {"desc": "示例 1 (混合岛屿)", "grid": [[0,0,1,0,0],[0,1,1,1,0],[0,0,0,0,0],[1,1,0,1,1]], "expected": 4},
        {"desc": "全是陆地", "grid": [[1,1],[1,1]], "expected": 4},
        {"desc": "全是水", "grid": [[0,0],[0,0]], "expected": 0},
        {"desc": "对角线孤岛 (不连通)", "grid": [[1,0,1],[0,0,0],[1,0,1]], "expected": 1},
        {"desc": "长条蛇形岛", "grid": [[1,1,1],[0,0,1],[1,1,1]], "expected": 7},
        {"desc": "单格岛", "grid": [[1]], "expected": 1},
        {"desc": "空网格", "grid": [], "expected": 0},
        {"desc": "只有一个大岛", "grid": [[1,1,1],[1,1,1]], "expected": 6},
        {"desc": "垂直长条", "grid": [[1],[1],[1]], "expected": 3},
        {"desc": "水平长条", "grid": [[1,1,1,1]], "expected": 4}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        grid_copy = copy.deepcopy(tc["grid"])
        actual = func(grid_copy)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")
    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_695(Solution695().maxAreaOfIsland)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 4    | 4    | 示例 1 (混合岛屿)
2    | ✅ PASS | 4    | 4    | 全是陆地
3    | ✅ PASS | 0    | 0    | 全是水
4    | ✅ PASS | 1    | 1    | 对角线孤岛 (不连通)
5    | ✅ PASS | 7    | 7    | 长条蛇形岛
6    | ✅ PASS | 1    | 1    | 单格岛
7    | ✅ PASS | 0    | 0    | 空网格
8    | ✅ PASS | 6    | 6    | 只有一个大岛
9    | ✅ PASS | 3    | 3    | 垂直长条
10   | ✅ PASS | 4    | 4    | 水平长条
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题和 200 题唯一的区别在于：**DFS 函数现在需要有返回值了。**

**1. 巡逻员（主循环）**：
   - 依然是遍历每一个格子。
   - 看到 `'1'` 时，不再只是 `count += 1`，而是问 DFS：“这块连通的岛屿总共有多大？”
   - **更新全局最大值**：`max_area = max(max_area, dfs(r, c))`。

**2. 拆迁队（DFS 递归函数） —— 重点！**：
   - **任务**：统计以 `(r, c)` 为起点的整块陆地面积。
   - **计算公式**：
     - 当前这一块陆地的面积是 `1`。
     - 总面积 = `1 + 向上搜到的面积 + 向下搜到的面积 + 向左搜到的面积 + 向右搜到的面积`。
   - **递归出口**：
     - 如果越界或者是水（`'0'`），说明这部分面积是 `0`。

**3. 细节**：
   - 别忘了“淹没”！为了不重复计算，每搜到一块陆地，依然要把它变成 `'0'`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N)$。每个格子最多被访问一次。
* **空间复杂度**：$O(M \times N)$。最坏情况下（全是陆地），递归栈深度等于面积。
:::

## 📝 题目 733：图像渲染 (Flood Fill)

::: {.callout-caution}
### 📖 题目描述
有一幅以 $m \times n$ 的整数矩阵表示的图画 `image` ，其中 `image[i][j]` 表示该像素点的颜色值。

给你三个整数：`sr` (行)、`sc` (列) 和 `newColor` 。你应该从像素 `image[sr][sc]` 开始对图像进行 **上色渲染** （Flood Fill）。

**渲染规则**：
1.  从起始像素 `image[sr][sc]` 开始。
2.  将起始像素的颜色改为 `newColor`。
3.  找出所有与起始像素 **相邻且颜色相同** 的像素点（4 个方向），并将它们的颜色也改为 `newColor`。
4.  继续对这些新上色的像素点重复上述过程，直到所有相连的同色区域都被染成 `newColor`。

最后返回经过上色渲染后的图像。

---

**输入输出示例**：

* **示例 1**：
    * **输入**：`image = [[1,1,1],[1,1,0],[1,0,1]]`, `sr = 1, sc = 1, newColor = 2`
    * **输出**：`[[2,2,2],[2,2,0],[2,0,1]]`
    * **解释**：在图像正中间，(1,1) 位置的颜色是 1。所有与之连通且颜色为 1 的点都被改成了 2。注意，右下角的 1 没有被改变，因为它不与起始点连通。

* **示例 2**：
    * **输入**：`image = [[0,0,0],[0,0,0]]`, `sr = 0, sc = 0, newColor = 2`
    * **输出**：`[[2,2,2],[2,2,2]]`

* **示例 3**：
    * **输入**：`image = [[0,0,0],[0,1,1]]`, `sr = 1, sc = 1, newColor = 1`
    * **输出**：`[[0,0,0],[0,1,1]]`
    * **解释**：起始颜色就是 1，新颜色也是 1，所以图像没有发生任何变化。
:::

In [33]:
class Solution733:
    def floodFill(self, image: List[List[int]], sr: int, sc: int, newColor: int) -> List[List[int]]:
        rows, cols = len(image), len(image[0])
        oldColor = image[sr][sc]
        if oldColor == newColor:
            return image

        def dfs(r: int, c: int) -> None:
            if 0 <= r < rows and 0 <= c < cols and image[r][c] == oldColor:
                image[r][c] = newColor
                dfs(r - 1, c)
                dfs(r + 1, c)
                dfs(r, c + 1)
                dfs(r, c - 1)

        dfs(sr, sc)
        return image

In [34]:
#| code-fold: true
def test_733(sol_func):
    test_cases = [
        {
            "desc": "示例 1: 标准连通扩散",
            "image": [[1,1,1],[1,1,0],[1,0,1]], "sr": 1, "sc": 1, "newColor": 2,
            "expected": [[2,2,2],[2,2,0],[2,0,1]]
        },
        {
            "desc": "示例 2: 全色块渲染",
            "image": [[0,0,0],[0,0,0]], "sr": 0, "sc": 0, "newColor": 2,
            "expected": [[2,2,2],[2,2,2]]
        },
        {
            "desc": "不连通色块测试 (关键!)",
            "image": [[1,0,1]], "sr": 0, "sc": 0, "newColor": 2,
            "expected": [[2,0,1]] # 右边的 1 不应该变
        },
        {
            "desc": "孤岛渲染",
            "image": [[0,0,0],[0,1,0],[0,0,0]], "sr": 1, "sc": 1, "newColor": 2,
            "expected": [[0,0,0],[0,2,0],[0,0,0]]
        },
        {
            "desc": "新旧颜色一致 (特判)",
            "image": [[1,1],[1,1]], "sr": 0, "sc": 0, "newColor": 1,
            "expected": [[1,1],[1,1]]
        },
        {
            "desc": "起始点在边界 (左上角)",
            "image": [[1,0],[0,0]], "sr": 0, "sc": 0, "newColor": 3,
            "expected": [[3,0],[0,0]]
        },
        {
            "desc": "起始点在边界 (右下角)",
            "image": [[0,0],[0,1]], "sr": 1, "sc": 1, "newColor": 3,
            "expected": [[0,0],[0,3]]
        },
        {
            "desc": "单像素渲染",
            "image": [[5]], "sr": 0, "sc": 0, "newColor": 2,
            "expected": [[2]]
        },
        {
            "desc": "复杂迷宫连通测试",
            "image": [[1,1,0],[0,1,0],[0,1,1]], "sr": 0, "sc": 0, "newColor": 8,
            "expected": [[8,8,0],[0,8,0],[0,8,8]]
        },
        {
            "desc": "十字架连通 (对角线不连通测试)",
            "image": [[0,1,0],[1,1,1],[0,1,0]], "sr": 1, "sc": 1, "newColor": 7,
            "expected": [[0,7,0],[7,7,7],[0,7,0]]
        }
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 50)
    for i, tc in enumerate(test_cases):
        img_copy = copy.deepcopy(tc["image"])
        actual = sol_func(img_copy, tc["sr"], tc["sc"], tc["newColor"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")

    print("-" * 50)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_733(Solution733().floodFill)

ID   | 结果     | 描述
--------------------------------------------------
1    | ✅ PASS | 示例 1: 标准连通扩散
2    | ✅ PASS | 示例 2: 全色块渲染
3    | ✅ PASS | 不连通色块测试 (关键!)
4    | ✅ PASS | 孤岛渲染
5    | ✅ PASS | 新旧颜色一致 (特判)
6    | ✅ PASS | 起始点在边界 (左上角)
7    | ✅ PASS | 起始点在边界 (右下角)
8    | ✅ PASS | 单像素渲染
9    | ✅ PASS | 复杂迷宫连通测试
10   | ✅ PASS | 十字架连通 (对角线不连通测试)
--------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题本质上是寻找 **“与起点颜色相同的所有连通分量”**。

**1. 记录初始状态**：
   - 首先获取起始点的原始颜色：`oldColor = image[sr][sc]`。
   - 这是我们判断“是否需要染色”的唯一标准。

**2. ⚠️ 核心陷阱：死循环预防**：
   - 如果 `oldColor` 和 `newColor` 是一样的，你就不需要做任何事。
   - 如果不加这个判断，DFS 会在同一个格子上反复横跳（改了颜色后发现颜色和目标一致，又触发递归），导致栈溢出。

**3. DFS 拆迁队任务**：
   - **检查边界**：是否跳出了地图。
   - **检查颜色**：当前格子颜色是否等于 `oldColor`？
   - **染色**：将当前格子的颜色修改为 `newColor`。
   - **扩散**：向 **上下左右** 四个方向发动递归。

**4. 为什么不用 visited 数组？**
   - 因为我们修改了原数组的颜色。只要 `newColor != oldColor`，修改后的格子就不再满足 `image[r][c] == oldColor` 的条件，自然就不会被二次访问。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N)$。最坏情况下需要遍历所有像素点。
* **空间复杂度**：$O(M \times N)$。递归调用栈的最大深度。
:::